In [1]:
# pip install pymongo faker pandas numpy
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from faker import Faker
from pymongo import MongoClient

SEED = 42
rng = np.random.default_rng(SEED)
Faker.seed(SEED); fake = Faker("en_IN")

MONGO_URI, DB_NAME, CSV_DIR = "mongodb://localhost:27017", "fitpulse", "data"
TRACKER_ADOPTION, SYNC_RATE = 0.80, 0.92

# ---------- load SQL-side extracts (same CSVs loaded into MySQL) ----------
members = pd.read_csv(f"{CSV_DIR}/members.csv", parse_dates=["date_of_birth"])
plans = pd.read_csv(f"{CSV_DIR}/plans.csv")
ms = pd.read_csv(f"{CSV_DIR}/memberships.csv", parse_dates=["start_date", "end_date"])
att = pd.read_csv(f"{CSV_DIR}/attendance.csv", parse_dates=["check_in_ts", "check_out_ts"])
bk = pd.read_csv(f"{CSV_DIR}/bookings.csv")
cl = pd.read_csv(f"{CSV_DIR}/classes.csv")
ct = pd.read_csv(f"{CSV_DIR}/class_types.csv")

att = (att.merge(bk[["booking_id", "class_id"]], on="booking_id", how="left")
          .merge(cl[["class_id", "class_type_id"]], on="class_id", how="left")
          .merge(ct[["class_type_id", "class_name"]], on="class_type_id", how="left"))

ref_date = att.check_in_ts.max().normalize()          # identical to @ref_date in queries.sql

# ---------- renewal_outcomes snapshot (definitions match SQL Q4) ----------
succ = set(ms.renewed_from_id.dropna().astype(int))
ms["renewed"] = ms.membership_id.isin(succ)
ms["eligible"] = ms.end_date <= ref_date - pd.Timedelta(days=7)
ms["churned"] = (~ms.renewed) & (ms.end_date < ref_date)
ms = ms.merge(plans[["plan_id", "plan_name"]], on="plan_id")
renewal_docs = [dict(
    membership_id=int(r.membership_id), member_id=int(r.member_id), plan_id=int(r.plan_id),
    plan_name=r.plan_name, start_date=r.start_date.to_pydatetime(),
    end_date=r.end_date.to_pydatetime(), price_paid=float(r.price_paid),
    renewed=bool(r.renewed), eligible=bool(r.eligible)) for r in ms.itertuples()]

# ---------- tracker-owner profiles ----------
DEVICES = [("Fitbit", "Charge 6", False), ("Garmin", "Forerunner 55", True),
           ("Apple", "Watch SE", True), ("Noise", "ColorFit Pro 4", False),
           ("Xiaomi", "Smart Band 8", False)]
prof = {}
for r in members.itertuples():
    if rng.random() > TRACKER_ADOPTION:
        continue
    brand, model, spo2 = DEVICES[int(rng.integers(len(DEVICES)))]
    base_w = {"Male": 78, "Female": 64}.get(r.gender, 70)
    prof[int(r.member_id)] = dict(
        age=int((ref_date - r.date_of_birth).days // 365),
        weight=float(np.clip(rng.normal(base_w, 11), 45, 120)),
        rest=int(np.clip(rng.normal(66, 7), 48, 90)),
        device=dict(device_id=fake.uuid4(), brand=brand, model=model,
                    serial=fake.bothify("SN-????-########").upper()),
        spo2=spo2)

# MET, steps/min, intensity range (fraction of heart-rate reserve)
PROFILES = {
    "Yoga": (3.0, 10, (.35, .55)), "HIIT": (8.5, 70, (.75, .92)), "Zumba": (6.5, 90, (.65, .85)),
    "Spin": (8.0, 0, (.70, .90)), "Pilates": (3.5, 12, (.40, .60)), "CrossFit": (8.0, 50, (.75, .95)),
    "Strength Training": (5.0, 25, (.55, .75)), "Boxing": (7.8, 60, (.70, .90)),
    "Meditation": (1.8, 3, (.25, .40)), "Aerobics": (6.5, 85, (.60, .80)),
    "Gym Floor": (5.5, 40, (.55, .80)), "Treadmill": (8.0, 130, (.65, .85)),
    "Outdoor Run": (9.5, 160, (.70, .90)), "Brisk Walk": (4.3, 110, (.45, .60)),
    "Cycling": (7.5, 0, (.60, .80))}
WALKIN, WALKIN_P = ["Gym Floor", "Treadmill", "Strength Training"], [.5, .3, .2]
OUTDOOR, OUTDOOR_P = ["Outdoor Run", "Brisk Walk", "Cycling"], [.3, .5, .2]
SPEED = {"Outdoor Run": (7, 11), "Brisk Walk": (4, 5.5), "Cycling": (14, 22)}

def make_log(m, mid, start, end, stype, source, booking_id=None):
    p = prof[m]; met, spm, (lo, hi) = PROFILES[stype]
    dur = max(10, int(round((end - start).total_seconds() / 60)))
    hr_cap = 220 - p["age"]
    avg_hr = int(p["rest"] + rng.uniform(lo, hi) * (hr_cap - p["rest"]))
    peak = int(min(hr_cap, avg_hr + int(rng.integers(8, 25))))
    low = int(max(p["rest"] + 5, avg_hr - int(rng.integers(15, 30))))
    steps = int(max(0, rng.normal(spm * dur, spm * dur * 0.12))) if spm else 0
    kcal = float(round(met * p["weight"] * dur / 60 * rng.uniform(0.92, 1.08), 1))
    doc = {
        "member_id": m, "membership_id": mid, "device": p["device"],
        "session_type": stype, "source": source,
        "start_ts": start, "end_ts": end, "duration_min": dur,
        "metrics": {"steps": steps, "calories_kcal": kcal,
                    "hr": {"avg": avg_hr, "max": peak, "min": low, "resting": p["rest"]}},
        "hr_samples": [{"t": t, "bpm": int(np.clip(rng.normal(avg_hr, 6), low, peak))}
                       for t in range(0, dur * 60, 300)],
        "tags": ["high_intensity"] if avg_hr > 0.8 * hr_cap else [],
        "ingested_at": datetime.now(timezone.utc)}
    if booking_id is not None:
        doc["booking_id"] = booking_id
    if stype in SPEED:
        doc["gps"] = {"distance_km": round(dur / 60 * float(rng.uniform(*SPEED[stype])), 2),
                      "elevation_gain_m": int(rng.integers(0, 120))}
    if p["spo2"]:
        doc["spo2_avg"] = int(rng.integers(94, 100))
    return doc

# ---------- membership windows per tracker member ----------
by_member = {m: g.sort_values("start_date") for m, g in ms.groupby("member_id")}
att_by_member = {m: g for m, g in att.groupby("member_id")}
wins = {}
for m in prof:
    a = att_by_member.get(m, att.iloc[0:0]); wins[m] = []
    for r in by_member.get(m, pd.DataFrame()).itertuples():
        eff_end = min(r.end_date, ref_date)
        weeks = max(1, (eff_end - r.start_date).days + 1) / 7
        n = ((a.check_in_ts >= r.start_date) &
             (a.check_in_ts < eff_end + pd.Timedelta(days=1))).sum()
        wins[m].append(dict(mid=int(r.membership_id), start=r.start_date, end=r.end_date,
                            eff_end=eff_end, churned=bool(r.churned), gym_rate=float(n) / weeks))

def find_window(m, ts):
    for w in wins[m]:
        if w["start"] <= ts < w["end"] + pd.Timedelta(days=1):
            return w
    return None

docs = []
# (a) gym visits mirrored from SQL attendance
for r in att[att.member_id.isin(prof.keys())].itertuples():
    if rng.random() > SYNC_RATE:
        continue
    m = int(r.member_id); w = find_window(m, r.check_in_ts)
    if w is None:
        continue
    stype = r.class_name if isinstance(r.class_name, str) else str(rng.choice(WALKIN, p=WALKIN_P))
    bid = int(r.booking_id) if pd.notna(r.booking_id) else None
    docs.append(make_log(m, w["mid"], r.check_in_ts.to_pydatetime(), r.check_out_ts.to_pydatetime(),
                         stype, "gym_checkin", bid))

# (b) outdoor sessions: rate scales with gym frequency; drops before churn
for m, ws in wins.items():
    for w in ws:
        out_week = 0.25 + 0.35 * w["gym_rate"]
        for i in range((w["eff_end"] - w["start"]).days + 1):
            day = w["start"] + pd.Timedelta(days=i)
            p = out_week / 7
            if w["churned"] and (w["end"] - day).days <= 28:
                p *= 0.4
            if rng.random() < p:
                hour = int(rng.choice([6, 7, 8, 17, 18, 19], p=[.25, .20, .05, .15, .25, .10]))
                s = (day + pd.Timedelta(hours=hour, minutes=int(rng.integers(0, 60)))).to_pydatetime()
                e = s + pd.Timedelta(minutes=int(np.clip(rng.normal(40, 12), 15, 100))).to_pytimedelta()
                docs.append(make_log(m, w["mid"], s, e, str(rng.choice(OUTDOOR, p=OUTDOOR_P)), "tracker_auto"))

docs.sort(key=lambda d: d["start_ts"])

# ---------- MongoDB load ----------
client = MongoClient(MONGO_URI); db = client[DB_NAME]
db.drop_collection("workout_logs"); db.drop_collection("renewal_outcomes")
db.create_collection("workout_logs", validationLevel="moderate", validationAction="error", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["member_id", "membership_id", "device", "session_type",
                     "start_ts", "end_ts", "duration_min", "metrics"],
        "properties": {
            "member_id": {"bsonType": ["int", "long"]},
            "membership_id": {"bsonType": ["int", "long"]},
            "start_ts": {"bsonType": "date"}, "end_ts": {"bsonType": "date"},
            "duration_min": {"bsonType": ["int", "long"], "minimum": 1},
            "metrics": {"bsonType": "object", "required": ["steps", "calories_kcal", "hr"],
                        "properties": {
                            "steps": {"bsonType": ["int", "long"], "minimum": 0},
                            "calories_kcal": {"bsonType": "double", "minimum": 0},
                            "hr": {"bsonType": "object", "required": ["avg", "max"],
                                   "properties": {"avg": {"bsonType": ["int", "long"], "minimum": 30, "maximum": 220},
                                                  "max": {"bsonType": ["int", "long"], "minimum": 30, "maximum": 230}}}}}}}})
db.renewal_outcomes.insert_many(renewal_docs)
for i in range(0, len(docs), 5000):
    db.workout_logs.insert_many(docs[i:i + 5000], ordered=False)

n = db.workout_logs.count_documents({})
assert n >= 500
print(f"workout_logs: {n:,} | renewal_outcomes: {db.renewal_outcomes.count_documents({}):,} "
      f"| tracker members: {len(prof)}")
print(pd.Series([d['session_type'] for d in docs]).value_counts().to_string())

workout_logs: 24,652 | renewal_outcomes: 314 | tracker members: 117
Gym Floor            3832
Brisk Walk           3584
Strength Training    2690
Treadmill            2279
Outdoor Run          2100
HIIT                 1760
Zumba                1699
Cycling              1414
Yoga                 1212
CrossFit             1111
Spin                 1083
Pilates               732
Aerobics              684
Boxing                287
Meditation            185
